In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# الخلية 1️⃣: إعداد البيئة والمكتبات
# Cell 1: Environment Setup and Libraries
# ═══════════════════════════════════════════════════════════════════════

import subprocess, sys, os

# ── التحقق من GPU ──────────────────────────────────────────────────────
print('🖥️  فحص البيئة / Checking environment...')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || print('⚠️ GPU غير متوفر / GPU not available')

import torch
print(f'⚡ PyTorch: {torch.__version__}')
print(f'🎮 CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🏷️  GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── تثبيت المكتبات القياسية ────────────────────────────────────────────
print('\n📦 تثبيت المكتبات / Installing libraries...')
!pip install -q \
    transformers==4.41.2 \
    opencv-python-headless \
    Pillow \
    numpy \
    matplotlib \
    rasterio \
    scipy \
    supervision \
    timm \
    PyYAML

# ── تثبيت GroundingDINO ────────────────────────────────────────────────
print('\n🦖 تثبيت Grounding DINO / Installing Grounding DINO...')
if not os.path.exists('/content/GroundingDINO'):
    !git clone -q https://github.com/IDEA-Research/GroundingDINO.git /content/GroundingDINO

# تطبيق الـ Patch لإصلاح مشكلة _C في Colab 2025
# Apply patch to fix _C issue in Colab 2025
patch_targets = [
    '/content/GroundingDINO/groundingdino/models/GroundingDINO/ms_deform_attn.py',
    '/content/GroundingDINO/groundingdino/models/GroundingDINO/csrc/vision.cpp',
]
for pt in patch_targets:
    if os.path.exists(pt):
        with open(pt, 'r') as f: content = f.read()
        # استبدال استدعاء _C بـ Python fallback
        content = content.replace(
            'from groundingdino import _C',
            '# from groundingdino import _C  # patched'
        )
        with open(pt, 'w') as f: f.write(content)

sys.path.insert(0, '/content/GroundingDINO')
!cd /content/GroundingDINO && pip install -q -e . --no-build-isolation 2>/dev/null || echo '⚠️ تحذير: فشل البناء، سنستخدم Python fallback'

# ── تثبيت SAM ──────────────────────────────────────────────────────────
print('\n🎯 تثبيت SAM / Installing SAM...')
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

print('\n✅ إعداد البيئة اكتمل! / Environment setup complete!')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# الخلية 2️⃣: تحميل الإعدادات
# Cell 2: Load Configuration Files
# ═══════════════════════════════════════════════════════════════════════

import yaml
import os
from pathlib import Path

# ── تحديد مسار المشروع ─────────────────────────────────────────────────
# في Colab، انسخ ملفات المشروع إلى /content/mash
# In Colab, copy project files to /content/mash
PROJECT_ROOT = Path('/content/mash')  # عدّل هذا المسار حسب الحاجة

# إذا كنت تعمل محلياً / If working locally:
# PROJECT_ROOT = Path('.').resolve()

CONFIGS_DIR = PROJECT_ROOT / 'configs'
UTILS_DIR   = PROJECT_ROOT / 'utils'

# إضافة المسارات لـ sys.path
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── تحميل ملفات الإعدادات ──────────────────────────────────────────────
def load_config(config_name: str) -> dict:
    """تحميل ملف إعداد YAML / Load a YAML config file"""
    config_path = CONFIGS_DIR / config_name
    if not config_path.exists():
        raise FileNotFoundError(f'ملف الإعداد غير موجود: {config_path}')
    with open(config_path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

model_cfg      = load_config('model_config.yaml')
prompt_cfg     = load_config('prompt_config.yaml')
processing_cfg = load_config('processing_config.yaml')

print('✅ تم تحميل ملفات الإعدادات / Config files loaded:')
print(f'   📋 model_config.yaml     - {len(model_cfg)} أقسام')
print(f'   📋 prompt_config.yaml    - {len(prompt_cfg)} أقسام')
print(f'   📋 processing_config.yaml - {len(processing_cfg)} أقسام')

# ── تهيئة مجلدات الإخراج ──────────────────────────────────────────────
OUTPUT_DIR = Path(model_cfg['environment']['output_dir'])
WEIGHTS_DIR = Path(model_cfg['environment']['weights_dir'])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'\n📁 مجلد الإخراج: {OUTPUT_DIR}')
print(f'📁 مجلد الأوزان: {WEIGHTS_DIR}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# الخلية 3️⃣: تحميل الصور من المجلد
# Cell 3: Load Images from Directory
# ═══════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import glob as _glob

from utils.data_loader import ImageLoader

# ── تحميل بيانات SpaceNet (أو استخدم مجلدك الخاص) ────────────────────
print('⬇️  تحميل بيانات SpaceNet...')
dataset_path = kagglehub.dataset_download('ugorjiir/spacenet-2-paris-buildings')
print(f'✅ البيانات في: {dataset_path}')

# الحصول على مسارات صور TIF
all_image_paths = _glob.glob(f'{dataset_path}/**/*.tif', recursive=True)
print(f'📸 عدد الصور المتاحة: {len(all_image_paths)}')

# استخدام أول 5 صور للتجربة (زد الرقم حسب حاجتك)
TEST_LIMIT = 5
test_image_paths = sorted(all_image_paths)[:TEST_LIMIT]

print(f'\n🎯 سيتم معالجة {len(test_image_paths)} صورة:')
for i, p in enumerate(test_image_paths):
    print(f'   [{i}] {os.path.basename(p)}')

# ── تهيئة ImageLoader ──────────────────────────────────────────────────
# استخدام أول صورة كتجربة
loader = ImageLoader(
    image_dir=os.path.dirname(test_image_paths[0]),
    max_size=processing_cfg['image_loading']['max_image_size'],
    normalize=processing_cfg['image_loading']['normalize'],
)

# تحميل الصور
loaded_images = []
for p in test_image_paths:
    try:
        img = loader.load_image(p)
        info = loader.get_image_info(p)
        loaded_images.append({'array': img, 'path': p, 'info': info})
        print(f'✅ {os.path.basename(p)}: {img.shape}')
    except Exception as e:
        print(f'❌ فشل تحميل {os.path.basename(p)}: {e}')

print(f'\n📦 تم تحميل {len(loaded_images)} صورة بنجاح')

# ── عرض عينة من الصور ─────────────────────────────────────────────────
n_show = min(3, len(loaded_images))
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 5))
if n_show == 1:
    axes = [axes]
for ax, img_data in zip(axes, loaded_images[:n_show]):
    img_disp = img_data['array']
    ax.imshow(np.clip(img_disp, 0, 1) if img_disp.max() <= 1 else img_disp)
    ax.set_title(os.path.basename(img_data['path'])[:20], fontsize=8)
    ax.axis('off')
plt.suptitle('عينة من الصور المحمّلة / Sample Loaded Images', fontsize=12)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'sample_images.png'), dpi=100, bbox_inches='tight')
plt.show()
print('✅ تم حفظ العينة')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# الخلية 4️⃣: كشف المباني باستخدام DINO
# Cell 4: Building Detection with DINO
# ═══════════════════════════════════════════════════════════════════════

from utils.dino_detector import DINODetector

# ── تحميل أوزان DINO ──────────────────────────────────────────────────
gdino_cfg = model_cfg['grounding_dino']
gdino_weights = WEIGHTS_DIR / 'gdino.pth'

if not gdino_weights.exists():
    print('⬇️  تحميل أوزان Grounding DINO...')
    !wget -q --show-progress \
        https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth \
        -O {gdino_weights}
else:
    print('✅ أوزان DINO موجودة')

# ── تهيئة نموذج DINO ──────────────────────────────────────────────────
dino_config = '/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'

detector = DINODetector(
    weights_path=str(gdino_weights),
    config_path=dino_config,
    box_threshold=gdino_cfg['box_threshold'],
    text_threshold=gdino_cfg['text_threshold'],
)

print('\n✅ نموذج DINO جاهز!')

# ── تشغيل الكشف على كل الصور ──────────────────────────────────────────
all_detections = []  # قائمة الكشوفات لكل صورة
prompt_variant = prompt_cfg['defaults']['prompt_variant']  # 'primary'

for img_data in loaded_images:
    img = img_data['array']
    img_name = os.path.basename(img_data['path'])
    print(f'\n🔍 كشف المباني في: {img_name}')

    try:
        # كشف المباني
        custom_prompt = prompt_cfg['prompts']['buildings'][prompt_variant]
        detections = detector.detect(
            image=img,
            custom_prompt=custom_prompt
        )

        # تصفية حسب الثقة
        detections = detector.filter_by_score(
            detections,
            min_score=processing_cfg['detection']['min_confidence']
        )

        print(f'   📦 عدد المباني المكتشفة: {detections["count"]}')

        all_detections.append({
            'image_data': img_data,
            'detections': detections
        })

        # عرض النتيجة
        if detections['count'] > 0:
            vis = detector.visualize_detections(img, detections)
            plt.figure(figsize=(10, 6))
            plt.imshow(vis)
            plt.title(f'DINO كشف المباني - {img_name}\nعدد المباني: {detections["count"]}')
            plt.axis('off')
            save_name = f'dino_detection_{os.path.splitext(img_name)[0]}.png'
            plt.savefig(str(OUTPUT_DIR / save_name), dpi=100, bbox_inches='tight')
            plt.show()

    except Exception as e:
        print(f'   ❌ خطأ: {e}')
        import traceback; traceback.print_exc()

total_detected = sum(d['detections']['count'] for d in all_detections)
print(f'\n✅ إجمالي المباني المكتشفة: {total_detected} في {len(all_detections)} صورة')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# الخلية 5️⃣: تقسيم الصور باستخدام SAM
# Cell 5: Image Segmentation with SAM
# ═══════════════════════════════════════════════════════════════════════

from utils.sam_segmenter import SAMSegmenter

# ── تحميل أوزان SAM ───────────────────────────────────────────────────
sam_cfg = model_cfg['sam']
sam_weights = WEIGHTS_DIR / 'sam_vit_h_4b8939.pth'

if not sam_weights.exists():
    print('⬇️  تحميل أوزان SAM (2.5 GB) - قد يستغرق بعض الوقت...')
    !wget -q --show-progress \
        https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth \
        -O {sam_weights}
else:
    print('✅ أوزان SAM موجودة')

# ── تهيئة نموذج SAM ───────────────────────────────────────────────────
segmenter = SAMSegmenter(
    checkpoint_path=str(sam_weights),
    model_type=sam_cfg['model_type'],
)
print('✅ نموذج SAM جاهز!')

# ── تشغيل التقسيم ──────────────────────────────────────────────────────
all_masks = []  # قائمة الأقنعة لكل صورة
seg_cfg = processing_cfg['segmentation']

for det_data in all_detections:
    img = det_data['image_data']['array']
    img_name = os.path.basename(det_data['image_data']['path'])
    detections = det_data['detections']

    if detections['count'] == 0:
        print(f'⚠️  {img_name}: لا توجد كشوفات')
        all_masks.append({'image_data': det_data['image_data'], 'masks': []})
        continue

    print(f'\n🎯 تقسيم {img_name} ({detections["count"]} مبنى)...')

    try:
        # تقسيم بناءً على صناديق DINO
        masks = segmenter.segment_from_boxes(
            image=img,
            boxes=detections['boxes'],
            scores=detections['scores'],
        )

        # دمج الأقنعة المتداخلة
        if len(masks) > 1:
            masks = segmenter.merge_overlapping_masks(
                masks,
                iou_threshold=seg_cfg['merge_iou_threshold']
            )

        print(f'   ✅ {len(masks)} قناع بعد الدمج')
        all_masks.append({'image_data': det_data['image_data'], 'masks': masks})

        # عرض عينة من الأقنعة
        if masks:
            img_disp = (img * 255).astype(np.uint8) if img.max() <= 1 else img.copy()
            overlay = img_disp.copy()
            rng = np.random.default_rng(42)
            colors = rng.integers(50, 255, size=(len(masks), 3))
            for m_idx, m in enumerate(masks):
                overlay[m['mask']] = (overlay[m['mask']] * 0.4 + colors[m_idx] * 0.6).astype(np.uint8)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
            ax1.imshow(img_disp); ax1.set_title('الأصلية'); ax1.axis('off')
            ax2.imshow(overlay); ax2.set_title(f'التقسيم ({len(masks)} مبنى)'); ax2.axis('off')
            plt.suptitle(f'SAM تقسيم - {img_name}')
            plt.tight_layout()
            save_name = f'sam_masks_{os.path.splitext(img_name)[0]}.png'
            plt.savefig(str(OUTPUT_DIR / save_name), dpi=100, bbox_inches='tight')
            plt.show()

    except Exception as e:
        print(f'   ❌ خطأ: {e}')
        import traceback; traceback.print_exc()
        all_masks.append({'image_data': det_data['image_data'], 'masks': []})

total_masks = sum(len(m['masks']) for m in all_masks)
print(f'\n✅ إجمالي الأقنعة: {total_masks}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# الخلية 6️⃣: المعالجة البعدية (Post-Processing)
# Cell 6: Post-Processing
# ═══════════════════════════════════════════════════════════════════════

from utils.post_processor import PostProcessor

# ── تهيئة المعالج البعدي ───────────────────────────────────────────────
pp_cfg = processing_cfg['post_processing']

processor = PostProcessor(
    min_area=pp_cfg['min_area'],
    max_area=pp_cfg['max_area'],
    smoothing_kernel=pp_cfg['smoothing_kernel'],
    closing_iterations=pp_cfg['closing_iterations'],
    opening_iterations=pp_cfg['opening_iterations'],
    fill_holes=pp_cfg['fill_holes'],
    remove_border_masks=pp_cfg['remove_border_masks'],
)
print('✅ المعالج البعدي جاهز')

# ── تطبيق المعالجة البعدية ─────────────────────────────────────────────
all_processed = []

for mask_data in all_masks:
    img = mask_data['image_data']['array']
    img_name = os.path.basename(mask_data['image_data']['path'])
    masks = mask_data['masks']

    if not masks:
        print(f'⚠️  {img_name}: لا توجد أقنعة')
        all_processed.append({'image_data': mask_data['image_data'], 'masks': []})
        continue

    print(f'\n🔧 معالجة {img_name} ({len(masks)} قناع)...')

    try:
        cleaned_masks = processor.process(masks)
        stats = processor.compute_statistics(cleaned_masks)
        seg_map = processor.create_segmentation_map(
            cleaned_masks,
            img.shape[:2]
        )

        print(f'   ✅ {stats["total_buildings"]} مبنى (مساحة متوسطة: {stats["avg_area_px"]:.0f} px²)')

        all_processed.append({
            'image_data': mask_data['image_data'],
            'masks': cleaned_masks,
            'seg_map': seg_map,
            'stats': stats,
        })

        # عرض خريطة التقسيم
        img_disp = (img * 255).astype(np.uint8) if img.max() <= 1 else img
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
        ax1.imshow(img_disp); ax1.set_title('الأصلية'); ax1.axis('off')

        overlay = img_disp.copy().astype(np.float32)
        rng = np.random.default_rng(42)
        colors = rng.integers(50, 255, size=(len(cleaned_masks), 3))
        for m_idx, m in enumerate(cleaned_masks):
            overlay[m['mask']] = overlay[m['mask']] * 0.4 + colors[m_idx] * 0.6
        ax2.imshow(overlay.astype(np.uint8)); ax2.set_title(f'بعد المعالجة ({stats["total_buildings"]} مبنى)'); ax2.axis('off')

        ax3.imshow(seg_map); ax3.set_title('خريطة التقسيم'); ax3.axis('off')
        plt.suptitle(f'المعالجة البعدية - {img_name}')
        plt.tight_layout()
        save_name = f'processed_{os.path.splitext(img_name)[0]}.png'
        plt.savefig(str(OUTPUT_DIR / save_name), dpi=100, bbox_inches='tight')
        plt.show()

    except Exception as e:
        print(f'   ❌ خطأ: {e}')
        import traceback; traceback.print_exc()
        all_processed.append({'image_data': mask_data['image_data'], 'masks': []})

total_cleaned = sum(len(p['masks']) for p in all_processed)
print(f'\n✅ إجمالي المباني بعد المعالجة: {total_cleaned}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# الخلية 7️⃣: توليد المعرفات الفريدة
# Cell 7: Generate Unique Building IDs
# ═══════════════════════════════════════════════════════════════════════

from utils.building_id_generator import BuildingIDGenerator
from pathlib import Path

# ── تهيئة مولّد المعرفات ───────────────────────────────────────────────
id_cfg = processing_cfg['id_generation']

id_gen = BuildingIDGenerator(
    prefix=id_cfg['prefix'],
    project_id=id_cfg['project_id'],
    use_uuid=id_cfg['use_uuid'],
)
print('✅ مولّد المعرفات جاهز')

# ── توليد المعرفات لكل المباني ─────────────────────────────────────────
for proc_data in all_processed:
    img_path = proc_data['image_data']['path']
    masks = proc_data.get('masks', [])

    if not masks:
        continue

    img_name = os.path.basename(img_path)
    print(f'\n🏢 توليد معرفات لـ {img_name} ({len(masks)} مبنى)...')

    building_ids = id_gen.generate_batch(
        masks=masks,
        image_path=img_path,
    )

    # تخزين المعرفات مع الأقنعة
    for mask, bid in zip(masks, building_ids):
        mask['building_id'] = bid

    print(f'   ✅ أول 3 معرفات: {building_ids[:3]}')

# ── حفظ البيانات ──────────────────────────────────────────────────────
output_json = OUTPUT_DIR / id_cfg['output_filename']
saved_path = id_gen.save_to_json(str(output_json))

summary = id_gen.get_summary()
print(f'\n📊 ملخص المباني المكتشفة:')
print(f'   🏢 إجمالي المباني: {summary["total_buildings"]}')
print(f'   🖼️  الصور المعالجة: {summary["processed_images"]}')
print(f'   📐 المساحة المتوسطة: {summary["area_stats"]["mean_px"]:.0f} px²')
print(f'   🎯 متوسط الثقة: {summary["confidence_stats"]["mean"]:.2%}')
print(f'   💾 البيانات محفوظة في: {saved_path}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# الخلية 8️⃣: عرض النتائج والإحصائيات
# Cell 8: Display Results and Statistics
# ═══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json

# ── تحميل البيانات المحفوظة ────────────────────────────────────────────
with open(str(output_json), 'r', encoding='utf-8') as f:
    buildings_data = json.load(f)

buildings = buildings_data['buildings']
metadata  = buildings_data['metadata']

print('═' * 60)
print(f'     📊 تقرير نتائج UrbanInsight')
print('═' * 60)
print(f'  المشروع:          {metadata["project_id"]}')
print(f'  وقت التشغيل:      {metadata["generated_at"]}')
print(f'  جلسة العمل:       {metadata["session_id"]}')
print(f'  إجمالي المباني:   {metadata["total_buildings"]}')
print('─' * 60)

# إحصائيات تفصيلية
if buildings:
    areas   = [b['geometry']['area_px'] for b in buildings]
    scores  = [b['confidence_score'] for b in buildings]
    widths  = [b['geometry']['bbox_width'] for b in buildings]
    heights = [b['geometry']['bbox_height'] for b in buildings]

    print(f'  مساحة أصغر مبنى: {min(areas):,} px²')
    print(f'  مساحة أكبر مبنى: {max(areas):,} px²')
    print(f'  المساحة المتوسطة: {np.mean(areas):.0f} px²')
    print(f'  متوسط الثقة:     {np.mean(scores):.1%}')
    print('═' * 60)

    # ── رسم الإحصائيات ────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 12))
    fig.patch.set_facecolor('#1a1a2e')
    gs = gridspec.GridSpec(2, 3, figure=fig)

    plot_style = dict(color='#4ecdc4', edgecolor='white', linewidth=0.5)
    title_style = dict(color='white', fontsize=11, pad=10)

    # توزيع المساحات
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_facecolor('#16213e')
    ax1.hist(areas, bins=20, **plot_style)
    ax1.set_title('توزيع مساحات المباني', **title_style)
    ax1.set_xlabel('المساحة (px²)', color='white')
    ax1.tick_params(colors='white')

    # توزيع درجات الثقة
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_facecolor('#16213e')
    ax2.hist(scores, bins=20, color='#ff6b6b', edgecolor='white', linewidth=0.5)
    ax2.set_title('توزيع درجات الثقة', **title_style)
    ax2.set_xlabel('درجة الثقة', color='white')
    ax2.tick_params(colors='white')

    # نسب الأبعاد
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.set_facecolor('#16213e')
    aspect_ratios = [b['geometry']['aspect_ratio'] for b in buildings]
    ax3.hist(aspect_ratios, bins=20, color='#ffd93d', edgecolor='white', linewidth=0.5)
    ax3.set_title('توزيع نسب الأبعاد', **title_style)
    ax3.set_xlabel('نسبة العرض/الارتفاع', color='white')
    ax3.tick_params(colors='white')

    # توزيع الأبعاد (عرض × ارتفاع)
    ax4 = fig.add_subplot(gs[1, 0:2])
    ax4.set_facecolor('#16213e')
    ax4.scatter(widths, heights, alpha=0.6, c=scores,
                cmap='viridis', s=30, edgecolors='none')
    ax4.set_title('أبعاد المباني (عرض × ارتفاع)', **title_style)
    ax4.set_xlabel('العرض (px)', color='white')
    ax4.set_ylabel('الارتفاع (px)', color='white')
    ax4.tick_params(colors='white')

    # ملخص الإحصائيات
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.set_facecolor('#16213e')
    ax5.axis('off')
    stats_text = (
        f'إجمالي المباني: {len(buildings)}\n'
        f'الصور المعالجة: {len(set(b["source_image"] for b in buildings))}\n'
        f'\n'
        f'المساحة الدنيا: {min(areas):,} px²\n'
        f'المساحة القصوى: {max(areas):,} px²\n'
        f'المساحة المتوسطة: {np.mean(areas):.0f} px²\n'
        f'\n'
        f'متوسط الثقة: {np.mean(scores):.1%}\n'
        f'أعلى ثقة: {max(scores):.1%}'
    )
    ax5.text(0.1, 0.9, stats_text, transform=ax5.transAxes,
             color='white', fontsize=11, verticalalignment='top',
             fontfamily='monospace')
    ax5.set_title('ملخص الإحصائيات', **title_style)

    plt.suptitle('📊 إحصائيات UrbanInsight - نتائج الكشف',
                 color='white', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'statistics_report.png'),
                dpi=120, bbox_inches='tight',
                facecolor='#1a1a2e')
    plt.show()
    print(f'\n💾 تم حفظ التقرير في: {OUTPUT_DIR / "statistics_report.png"}')

print('\n🎉 اكتملت المعالجة! / Processing complete!')
print(f'📁 جميع النتائج في: {OUTPUT_DIR}')
